# B1.10 · Exploit chaining

**Function B — Application Security with an AI SDLC → The AI SDLC: an Agentic AppSec Pipeline**  ·  *AI for Security*

Builds on **[B1.9 · Dynamic exploitation (DAST)](https://spbreed.github.io/cyber-commons/lessons/B1.9.html)**.

| | |
|---|---|
| Tools used | OWASP ZAP, Kimi K2, Claude Opus 5 |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off — and where a lesson involves a model, the same code calls an open-weight endpoint or a frontier API when you configure one.

## 1 · The hook

Three medium findings, each correctly scored, each individually not worth an engineer's afternoon. Chained, they read a file that ends the conversation about severity. Chains are where automated analysis earns its keep.

> **At CyberTravels.** Verbose errors, an open redirect and a path traversal are each low on their own. Chained against CyberTravels they read a config file and end the conversation about severity. R9.

## 2 · The framework

```
   alone                                chained
   +----------------+                   +-------------------------+
   | path traversal | medium            | traversal reads config  |
   | verbose errors | low        --->   | errors leak the key     |
   | open redirect  | low               | redirect delivers it    |
   +----------------+                   +-------------------------+
                                        outcome: credential exfiltration

   severity is a property of the chain, not of the link
```

**Stage 13 — Exploit chaining.** Individual findings are triaged individually,
and that is how three mediums become a critical nobody noticed.

The arithmetic of severity is not additive. A read-only information disclosure
is a medium. A CSRF is a medium. An unauthenticated internal endpoint is a
medium. Chained — leak an ID, forge a request using it, hit the internal
endpoint with the forged session — the outcome is account takeover, which is
not a medium.

The pipeline can find these mechanically because Phase 4 already produced
confirmed findings with known **preconditions** and **effects**. If one
finding's effect satisfies another's precondition, they compose, and the chain's
severity is the severity of its final effect.

This is the stage that most often changes what gets fixed first.

> **Where you are in the pipeline.**
>
> ```
> [Ingestion & Mapping] ──> [Threat Modelling] ──> [Discovery]
>          └─ stages 1-4         └─ stages 5-6        └─ stages 7-10
>                    ──> [Dynamic Validation] ──> [Reporting]
>                              └─ stages 11-14        └─ stage 15
> ```

## 3 · Stage 13 — findings as preconditions and effects

In [ ]:
from dataclasses import dataclass, field
from itertools import permutations

@dataclass(frozen=True)
class Confirmed:
    fid: str; cwe: str; name: str
    requires: frozenset       # preconditions
    grants: frozenset         # effects
    severity: str

FINDINGS = [
 Confirmed("F-01","CWE-200","report id disclosed in an error message",
           frozenset({"unauthenticated"}), frozenset({"valid_report_id"}), "low"),
 Confirmed("F-02","CWE-639","report fetch does not check ownership (IDOR)",
           frozenset({"valid_report_id","authenticated"}),
           frozenset({"other_users_report_data"}), "medium"),
 Confirmed("F-03","CWE-352","password change endpoint lacks CSRF protection",
           frozenset({"authenticated"}), frozenset({"password_reset_for_victim"}), "medium"),
 Confirmed("F-04","CWE-89","SQL injection in the report filter",
           frozenset({"authenticated"}), frozenset({"database_read"}), "high"),
 Confirmed("F-05","CWE-306","internal admin endpoint has no authentication",
           frozenset({"internal_network"}), frozenset({"admin_actions"}), "medium"),
 Confirmed("F-06","CWE-918","report export fetches a user-supplied URL (SSRF)",
           frozenset({"authenticated"}), frozenset({"internal_network"}), "medium"),
]
SEV_RANK = {"low":1,"medium":2,"high":3,"critical":4}
START = frozenset({"unauthenticated","authenticated"})

print(f"{'id':7s}{'cwe':10s}{'sev':9s}{'requires':38s}grants")
print("-" * 96)
for f in FINDINGS:
    print(f"{f.fid:7s}{f.cwe:10s}{f.severity:9s}{str(sorted(f.requires)):38s}"
          f"{sorted(f.grants)}")

## 4 · Compose them — an effect that satisfies the next precondition

In [ ]:
EFFECT_SEVERITY = {
 "other_users_report_data": "high",
 "password_reset_for_victim": "critical",
 "admin_actions": "critical",
 "database_read": "high",
 "internal_network": "medium",
 "valid_report_id": "low",
}

def chains(findings, start, max_len=4):
    found = []
    def walk(path, state):
        if len(path) >= max_len: return
        for f in findings:
            if f in path: continue
            if not f.requires <= state: continue
            new_state = state | f.grants
            new_path = path + [f]
            if len(new_path) > 1:
                worst = max((EFFECT_SEVERITY.get(g, "low") for g in f.grants),
                            key=lambda s: SEV_RANK[s])
                found.append({"chain": new_path, "final_effect": sorted(f.grants),
                              "severity": worst})
            walk(new_path, new_state)
    walk([], start)
    return found

ALL = chains(FINDINGS, START)
best = {}
for c in ALL:
    key = tuple(f.fid for f in c["chain"])
    best[key] = c
ranked = sorted(best.values(), key=lambda c: (-SEV_RANK[c["severity"]], len(c["chain"])))

print(f"{len(ranked)} composable chains found\n")
for c in ranked[:6]:
    ids = " → ".join(f.fid for f in c["chain"])
    links = ", ".join(f"{f.severity}" for f in c["chain"])
    print(f"[{c['severity']:8s}] {ids:26s} links: {links}")
    print(f"{'':11s} final effect: {c['final_effect']}")

## 5 · Where it breaks — triage the links, miss the chain

In [ ]:
individual = max(SEV_RANK[f.severity] for f in FINDINGS)
chained = max(SEV_RANK[c["severity"]] for c in ranked)
inv = {v: k for k, v in SEV_RANK.items()}
print(f"highest individual finding severity : {inv[individual]}")
print(f"highest chained severity            : {inv[chained]}")

critical_chains = [c for c in ranked if c["severity"] == "critical"]
print(f"\ncritical chains built entirely from non-critical findings:")
for c in critical_chains[:3]:
    ids = " → ".join(f"{f.fid}({f.severity})" for f in c["chain"])
    print(f"   {ids}")
    print(f"      → {c['final_effect']}")

all_links_medium_or_below = [c for c in critical_chains
                             if all(SEV_RANK[f.severity] <= 2 for f in c["chain"])]
print(f"\n{len(all_links_medium_or_below)} critical chain(s) whose every link is "
      f"medium or lower.")
print("Triaged individually, none of those findings would be worked this sprint.")
assert all_links_medium_or_below

In [ ]:
# The control: rank by chain severity, and report the chain, not the link.
def remediation_order(findings, chains_):
    """Fixing one link breaks every chain through it. Rank by chains broken."""
    impact = {}
    for f in findings:
        broken = [c for c in chains_ if f in c["chain"]]
        worst = max((SEV_RANK[c["severity"]] for c in broken), default=0)
        impact[f.fid] = {"chains_broken": len(broken), "worst_chain": inv.get(worst, "—"),
                         "own_severity": f.severity}
    return sorted(impact.items(),
                  key=lambda kv: (-SEV_RANK.get(kv[1]["worst_chain"], 0),
                                  -kv[1]["chains_broken"]))

print(f"{'finding':9s}{'own sev':10s}{'chains broken':>15}{'worst chain':>14}")
print("-" * 50)
for fid, i in remediation_order(FINDINGS, ranked):
    print(f"{fid:9s}{i['own_severity']:10s}{i['chains_broken']:>15}{i['worst_chain']:>14}")
top = remediation_order(FINDINGS, ranked)[0]
print(f"\nfix first: {top[0]} — own severity {top[1]['own_severity']}, but it "
      f"breaks {top[1]['chains_broken']} chains including a {top[1]['worst_chain']}")

## 6 · Phase 4 as a skill — and the preconditions that gate it

Dynamic validation is the one phase that *acts*. Everything before it reads; this one sends input to a running system. The skill therefore opens with safety preconditions rather than a procedure, and a refusal is a first-class output.

The contract also insists that `reproduced: false` be reported rather than dropped. A finding that survived Phase 3 and then failed to reproduce is the most useful signal the pipeline produces about its own false-positive rate — and it is the one a tidy report deletes.

In [ ]:
import json, re

def parse_skill(md):
    """Split a SKILL.md into (frontmatter dict, body).

    Frontmatter is a small, fixed subset of YAML: `key: value`, plus folded
    scalars (`description: >-`) whose continuation lines are indented. That is
    all a skill needs, and parsing it directly means no dependency.
    """
    if not md.startswith("---"):
        raise ValueError("a SKILL.md must open with a frontmatter block")
    _, front, body = md.split("---", 2)
    meta, key = {}, None
    for line in front.strip().splitlines():
        if not line.strip():
            continue
        if not line[0].isspace() and ":" in line:
            key, val = line.split(":", 1)
            key, val = key.strip(), val.strip()
            # `>-` and `|` open a folded block; the value is on the next lines
            meta[key] = "" if val in (">-", ">", "|", "|-") else val
        elif key is not None:
            meta[key] = (meta[key] + " " + line.strip()).strip()
    if "allowed-tools" in meta:
        meta["allowed-tools"] = [t.strip() for t in meta["allowed-tools"].split(",")
                                 if t.strip()]
    for required in ("name", "description"):
        if not meta.get(required):
            raise ValueError(f"skill is missing a {required!r}")
    return meta, body.strip()

_WORD = re.compile(r"[a-z][a-z-]{3,}")

def route(task, skills):
    """Pick the skill whose description best matches a task. Deterministic.

    The description is not documentation — it is the routing key. An agent
    decides whether to load a skill by reading it, so a vague description means
    the skill never fires when it should, and two overlapping descriptions mean
    the wrong one fires.

    Returns (pick, scores, margin). A margin of 0 means the top two scored the
    same and the "winner" is just whichever sorted first — an arbitrary answer
    wearing a confident face. Callers should refuse to auto-route on margin 0
    rather than pretend the tiebreak meant something.
    """
    want = set(_WORD.findall(task.lower()))
    def score(meta):
        return len(want & set(_WORD.findall(meta["description"].lower())))
    scores = {n: score(skills[n]) for n in sorted(skills)}
    # sort names first, then by score: ties must break identically on every
    # machine or the same task routes differently on two runs
    ranked = sorted(sorted(skills), key=lambda n: -scores[n])
    top = scores[ranked[0]]
    margin = top - (scores[ranked[1]] if len(ranked) > 1 else 0)
    return ranked[0], scores, margin

def contract_of(body):
    """The JSON block under '## Output contract' — the skill's machine promise."""
    # non-greedy across any prose between the heading and the fence
    m = re.search(r"## Output contract\b.*?```json\n(.*?)```", body, re.S)
    if not m:
        raise ValueError("skill declares no output contract")
    return json.loads(m.group(1))

def check(instance, contract, path="$"):
    """Structural conformance of an instance against a contract template.

    Returns the list of problems. An empty list means the shape is right — and
    that is *all* it means. Conformance is not accuracy: an empty findings list
    conforms perfectly and tells you nothing.
    """
    problems = []
    if isinstance(contract, dict):
        if not isinstance(instance, dict):
            return [f"{path}: expected an object, got {type(instance).__name__}"]
        for k, v in sorted(contract.items()):
            if k not in instance:
                problems.append(f"{path}.{k}: missing")
            else:
                problems += check(instance[k], v, f"{path}.{k}")
    elif isinstance(contract, list):
        if not isinstance(instance, list):
            return [f"{path}: expected a list, got {type(instance).__name__}"]
        for i, item in enumerate(instance):          # every element, same template
            problems += check(item, contract[0], f"{path}[{i}]")
    elif isinstance(contract, str) and "|" in contract:
        if instance not in contract.split("|"):
            problems.append(f"{path}: {instance!r} is not one of {contract}")
    elif isinstance(contract, bool):                  # before the numeric case:
        if not isinstance(instance, bool):            # bool is a subclass of int
            problems.append(f"{path}: expected bool, got {type(instance).__name__}")
    elif isinstance(contract, (int, float)):
        # JSON has one number type. A contract written `0` must accept 0.4, or
        # every cost and rate in the pipeline has to be rounded to satisfy a
        # checker rather than to be correct.
        if isinstance(instance, bool) or not isinstance(instance, (int, float)):
            problems.append(f"{path}: expected a number, got {type(instance).__name__}")
    elif not isinstance(instance, type(contract)):
        problems.append(f"{path}: expected {type(contract).__name__}, "
                        f"got {type(instance).__name__}")
    return problems

In [ ]:
# skills/appsec/appsec-exploit-validate/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: appsec-exploit-validate
description: >-
  Prove a candidate vulnerability is real by reproducing it in an isolated
  sandbox, then decide whether it chains into something worse. Use when asked
  to validate or confirm a finding, write or run a proof-of-concept, check
  exploitability, reproduce a CVE, or separate theoretical findings from
  demonstrated ones.
allowed-tools: Bash, Read, Write, Grep
---

# AppSec pipeline · Phase 4 — Dynamic validation

Covers **stages 11–14**. A finding that has been reproduced is a different
object from a finding that has been argued for. This phase produces the
difference, and it is the phase with the sharpest safety constraints.

## When to use this

Only on findings that reached `feasible: true, verdict: confirmed` in Phase 3.
Validating an unfiltered list wastes the most expensive stage in the pipeline
on findings Phase 3 would have deleted.

## Safety preconditions — check before anything else

Refuse to proceed unless **all** hold, and say which one failed:

1. The target is a **local sandbox or an explicitly authorised environment**.
   Never a production host, never a third party's infrastructure.
2. The sandbox has **no credentials** beyond throwaway ones, and **no route**
   to a real network the exploit could reach.
3. The proof-of-concept **demonstrates** the flaw. It does not exfiltrate real
   data, persist, escalate beyond the demonstration, or damage anything.
4. There is a **teardown** and it runs even on failure.

This is not ceremony. A validation harness with production credentials in its
environment is itself the vulnerability.

## Procedure

**Stage 11 — Sandbox replication.** Stand the component up in isolation with
the smallest fixture that reproduces the conditions: the vulnerable version,
the reachable entry point, and nothing else. Record the exact commit and the
fixture, because a PoC that cannot be re-run is an anecdote.

**Stage 12 — Dynamic exploitation.** Drive the entry point with an input that
should trigger the sink. Capture the observable: the crash, the query executed,
the file read, the process spawned. **The observable is the evidence** — an
exit code is not. Record what you sent and what came back.

**Stage 13 — Exploit chaining.** Ask what the demonstrated primitive gives
access to next. A path traversal that reads a config file containing a token is
not a file-read bug; it is a credential-disclosure bug. Chain only within the
sandbox, and stop at the first step that would need a real credential.

**Stage 14 — Remediation engineering.** Propose the fix at the right layer, and
say what it costs. Prefer the control that makes the class impossible
(parameterised queries, an allowlist, a type) over the one that blocks the
sample payload (a regex on the input you happened to send). Then **re-run
stage 12 against the fix** — a remediation that has not been tested against the
PoC that motivated it is a hypothesis.

## Output contract

```json
{
  "validations": [
    {"finding_id": "str", "reproduced": true,
     "sandbox": {"commit": "str", "fixture": "str", "isolated": true},
     "input": "str", "observable": "str",
     "chain": [{"primitive": "str", "leads_to": "str", "stopped_because": "str"}],
     "remediation": {"layer": "input|query|api|config|architecture",
                     "change": "str", "cost": "low|medium|high",
                     "retested": true, "still_reproduces": false}}
  ],
  "refused": [{"finding_id": "str", "precondition_failed": 1, "why": "str"}]
}
```

`reproduced: false` is a first-class result and must be reported, not dropped.
A finding that survived Phase 3 and then failed to reproduce is the single most
useful signal the pipeline produces about its own false-positive rate.

## Failure modes

- **Reporting `reproduced: true` without an observable.** The observable is the
  claim; without it there is nothing to check.
- **Treating "the exploit ran" as "the fix works".** Re-run after remediation
  or set `retested: false` and say so.
- **Chaining outside the sandbox.** Stop and record `stopped_because`.
- **Fixing the payload instead of the class.** A regex that blocks `../` does
  not fix path traversal.

## Handoff

Pass `validations` to **appsec-triage-report**. Findings that failed to
reproduce keep their finding record and gain `reproduced: false` — the report
must be able to distinguish "demonstrated" from "asserted".
"""

meta, body = parse_skill(SKILL_MD)
print(f"loaded skill: {meta['name']}")
print(f"  tools it may use: {', '.join(meta.get('allowed-tools', [])) or '—'}")
print(f"  routing description: {len(meta['description'].split())} words")
print(f"  procedure: {len(body.splitlines())} lines")

In [ ]:
contract = contract_of(body)

def validation_of(f):
    """One confirmed finding, expressed as the skill's validation record."""
    return {
      "finding_id": f.fid, "reproduced": True,
      "sandbox": {"commit": "a1fcf68", "fixture": f"minimal app exposing {f.fid}",
                  "isolated": True},
      "input": f"crafted request exercising {f.cwe}",
      # the observable is the claim; an exit code is not
      "observable": f.name,
      "chain": [{"primitive": f.name, "leads_to": g,
                 "stopped_because": "next step needs a real credential"}
                for g in sorted(f.grants)],
      "remediation": {"layer": "query" if f.cwe == "CWE-89" else "api",
                      "change": f"eliminate the class behind {f.cwe}",
                      "cost": "low", "retested": True, "still_reproduces": False},
    }

phase4 = {
 "validations": [validation_of(f) for f in FINDINGS],
 # the safety gate, exercised rather than described
 "refused": [{"finding_id": "F-99", "precondition_failed": 1,
              "why": "target is a production host, not a sandbox"}],
}
problems = check(phase4, contract)
print(f"conformance: {len(problems)} problem(s)")
for p in problems: print("   ", p)
assert not problems, problems

print(f"\nvalidated {len(phase4['validations'])} findings, "
      f"refused {len(phase4['refused'])}")
for v in phase4["validations"][:3]:
    print(f"   {v['finding_id']}  chains to: "
          f"{', '.join(c['leads_to'] for c in v['chain']) or '—'}")
print()
print("Refusing is an output, not an error. A validation harness that quietly")
print("skips the production target reports the same thing as one that never")
print("saw it, and those are very different states to be in.")
assert phase4["refused"], "the safety gate must be visible in the output"

## 7 · Where it breaks — the tidy report

Now suppose two of these findings do not reproduce, and the pipeline does the natural thing with them.

In [ ]:
mixed = [dict(validation_of(f), reproduced=(i % 3 != 0))
         for i, f in enumerate(FINDINGS)]
failed = [v for v in mixed if not v["reproduced"]]

tidy = {"validations": [v for v in mixed if v["reproduced"]], "refused": []}
honest = {"validations": mixed, "refused": phase4["refused"]}

print(f"conformance, tidy report  : {len(check(tidy, contract))}")
print(f"conformance, honest report: {len(check(honest, contract))}")
print(f"\nvalidations attempted : {len(mixed)}")
print(f"reproduced            : {len(mixed) - len(failed)}")
print(f"failed to reproduce   : {len(failed)}  ({', '.join(v['finding_id'] for v in failed)})")
fpr = len(failed) / len(mixed)
print(f"measured false-positive rate of Phase 3: {fpr:.0%}")
print()
print("Both conform. The tidy one drops the non-reproductions, and with them")
print("the only number that says how good the earlier phases actually are.")
print("Its reader sees a pipeline that is right every time.")
assert not check(tidy, contract), "dropping the failures is schema-valid - that is the point"
assert failed, "the demo needs at least one non-reproduction"

## What you just proved

Six confirmed findings compose into multiple chains. The highest individual severity is high while the highest chained severity is critical, and at least one critical chain is built entirely from medium-or-lower links — for example SSRF granting internal network access, then the unauthenticated admin endpoint. Remediation ordering puts a medium finding first because it breaks the most chains.

## Your turn

Take your current open findings and write down each one's preconditions and effects. The chaining falls out mechanically, and the finding you should fix first is usually not the one at the top of the severity-sorted queue.

---

**Next → [B1.11 · Remediation engineering](https://spbreed.github.io/cyber-commons/lessons/B1.11.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B1.10.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B1.10.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*